# LLaMA 3.1 8B Price Inference via QLoRA Adapters
Loads a quantized LLaMA 3.1 8B base model, attaches pre-trained QLoRA adapters, and evaluates price prediction accuracy on a held-out Amazon product test set. Predictions use top-K probability-weighted token averaging for more stable outputs than greedy decoding.

## Environment Verification

In [ ]:
# Confirm runtime environment before installing packages
import sys
import torch

print(f"Python  : {sys.version}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}  |  Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Inference will be very slow on CPU.")

In [ ]:
!pip install -q --upgrade transformers==4.48.3 accelerate==1.3.0 datasets==3.2.0
!pip install -q --upgrade peft==0.14.0 trl==0.14.0 bitsandbytes==0.46.0
!pip install -q --upgrade matplotlib scipy scikit-learn
!pip install -q --upgrade "huggingface_hub<1.0,>=0.24.0"

In [ ]:
import os
import re
import math
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
from huggingface_hub import login
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed
)
from datasets import load_dataset
from peft import PeftModel

%matplotlib inline

## Configuration
All tunable parameters and constants are declared here. Change model identifiers or evaluation size in this single cell.

In [ ]:
# Model and dataset identifiers
BASE_MODEL      = "meta-llama/Meta-Llama-3.1-8B"
DATASET_NAME    = "ed-donner/pricer-data"
FINETUNED_MODEL = "ed-donner/pricer-2024-09-13_13.04.39"
REVISION        = "e8d637df551603dc86cd7a1598a8f44af4d7ae36"

# Inference settings
TOP_K     = 3    # Number of candidate next-tokens used in weighted price averaging
TEST_SIZE = 250  # Number of test items to evaluate

# ANSI color codes for per-prediction quality output
GREEN    = "\033[92m"
YELLOW   = "\033[93m"
RED      = "\033[91m"
RESET    = "\033[0m"
COLOR_MAP = {"red": RED, "orange": YELLOW, "green": GREEN}

In [ ]:
# Authenticate with Hugging Face to access gated models and private datasets
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

if not hf_token:
    raise ValueError("HF_TOKEN not found in Colab secrets. Add it via the key icon in the sidebar.")

login(hf_token, add_to_git_credential=True)
print("Authenticated with Hugging Face.")

## Dataset Loading

In [ ]:
print(f"Fetching dataset: {DATASET_NAME}")
dataset = load_dataset(DATASET_NAME)
train   = dataset['train']
test    = dataset['test']

print(f"  Train: {len(train):,} examples")
print(f"  Test : {len(test):,} examples")

In [ ]:
# Inspect a single test record to confirm schema and content
sample = test[0]
print("--- Sample test record ---")
print(f"Prompt (first 200 chars): {sample['text'][:200]}...")
print(f"Ground truth price      : ${sample['price']:.2f}")

## Model Loading with 4-bit Quantization
NF4 quantization (via bitsandbytes) reduces LLaMA 3.1 8B VRAM usage from ~32 GB to ~5-6 GB, making it runnable on a T4 GPU. Double quantization further compresses the quantization constants themselves.

In [ ]:
# 4-bit NF4 config with double quantization and bfloat16 compute dtype
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)
print("Quantization config: 4-bit NF4 + double quant + bfloat16")

In [ ]:
print(f"Loading base model: {BASE_MODEL}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

footprint_gb = base_model.get_memory_footprint() / 1e9
print(f"Base model loaded  |  VRAM footprint: {footprint_gb:.2f} GB")

## Attaching QLoRA Adapters
The adapter weights trained via QLoRA are merged on top of the frozen quantized base model using PEFT's `PeftModel`.

In [ ]:
print(f"Attaching adapters: {FINETUNED_MODEL}")
print(f"Revision          : {REVISION}")

fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    FINETUNED_MODEL,
    revision=REVISION
)

total_gb = fine_tuned_model.get_memory_footprint() / 1e9
print(f"Fine-tuned model ready  |  Total VRAM: {total_gb:.2f} GB")

## Price Extraction Utility
Parses the model's raw text output and extracts a numeric price.

In [ ]:
def extract_price(text):
    """
    Extract a dollar amount from a string containing 'Price is $...'.
    Returns 0.0 if no valid number is found.
    """
    if "Price is $" not in text:
        return 0.0
    content = text.split("Price is $")[1].replace(',', '').replace('$', '')
    match   = re.search(r"[-+]?\d*\.?\d+", content)
    return float(match.group()) if match else 0.0

In [ ]:
# Validate extract_price against known inputs
test_cases = [
    ("Price is $24.99",                   24.99),
    ("Price is $1,234.50",              1234.50),
    ("Price is $a fabulous 899.99 or so", 899.99),
    ("No price here",                       0.0),
]

all_pass = True
for text, expected in test_cases:
    result = extract_price(text)
    status = "PASS" if result == expected else "FAIL"
    if status == "FAIL":
        all_pass = False
    print(f"[{status}]  '{text}'  ->  ${result:.2f}  (expected ${expected:.2f})")

print(f"\nAll tests passed: {all_pass}")

## Inference: Top-K Weighted Price Prediction
Rather than greedily decoding the single most-likely next token, this function examines the top-K candidate tokens, filters for numeric ones, and returns a probability-weighted average. This reduces sensitivity to borderline token probabilities and produces smoother price estimates.

In [ ]:
def predict_price(prompt, top_k=TOP_K):
    """
    Predict the price of a product from its text prompt.

    Runs a single forward pass to obtain next-token logits, then computes
    a softmax-probability-weighted average over the top-K numeric tokens.
    Falls back to 0.0 if no numeric tokens appear in the top-K candidates.

    Args:
        prompt:  The full product prompt ending with 'Price is $'.
        top_k:   Number of candidate tokens to consider.

    Returns:
        Estimated price as a float.
    """
    set_seed(42)
    inputs         = tokenizer.encode(prompt, return_tensors="pt").to("cuda")
    attention_mask = torch.ones(inputs.shape, device="cuda")

    with torch.no_grad():
        logits = fine_tuned_model(inputs, attention_mask=attention_mask).logits

    # Extract probabilities for only the final token position
    next_token_probs          = F.softmax(logits[:, -1, :].to("cpu"), dim=-1)
    top_probs, top_token_ids  = next_token_probs.topk(top_k)

    prices, weights = [], []
    for i in range(top_k):
        token = tokenizer.decode(top_token_ids[0][i])
        prob  = top_probs[0][i]
        try:
            price = float(token)
            if price > 0:
                prices.append(price)
                weights.append(prob)
        except ValueError:
            continue  # Skip non-numeric tokens

    if not prices:
        return 0.0

    total_weight = sum(weights)
    return sum(p * w / total_weight for p, w in zip(prices, weights)).item()

## Evaluation Framework
Three complementary metrics assess prediction quality:
- **MAE (Mean Absolute Error):** Average dollar difference between prediction and truth.
- **RMSLE:** Root Mean Squared Log Error — penalizes large relative errors more than large absolute ones.
- **Hit Rate:** Fraction of predictions within $40 or 20% of the true price (green zone).

In [ ]:
class Tester:
    """
    Evaluates any price predictor function against a labeled dataset.

    Tracks per-item predictions, errors, and log errors, then renders
    a summary report and a predicted-vs-actual scatter plot.
    """

    def __init__(self, predictor, data, title=None, size=TEST_SIZE):
        self.predictor = predictor
        self.data      = data
        self.title     = title or predictor.__name__.replace("_", " ").title()
        self.size      = min(size, len(data))
        self.guesses   = []
        self.truths    = []
        self.errors    = []
        self.sles      = []
        self.colors    = []

    def color_for(self, error, truth):
        """Classify prediction quality as green, orange, or red."""
        if error < 40 or error / truth < 0.2:
            return "green"
        if error < 80 or error / truth < 0.4:
            return "orange"
        return "red"

    def run_datapoint(self, i):
        """Run one prediction and record all metrics for index i."""
        datapoint = self.data[i]
        guess     = self.predictor(datapoint["text"])
        truth     = datapoint["price"]
        error     = abs(guess - truth)
        log_error = math.log(truth + 1) - math.log(guess + 1)
        sle       = log_error ** 2
        color     = self.color_for(error, truth)
        label     = datapoint["text"].split("\n\n")[1][:30] + "..."

        self.guesses.append(guess)
        self.truths.append(truth)
        self.errors.append(error)
        self.sles.append(sle)
        self.colors.append(color)

        print(
            f"{COLOR_MAP[color]}{i+1:>3}: "
            f"Predicted ${guess:>8,.2f}  |  Actual ${truth:>8,.2f}  "
            f"|  Error ${error:>7,.2f}  |  SLE {sle:.3f}  |  {label}{RESET}"
        )

    def chart(self, title):
        """Scatter plot of predicted vs. actual prices with ideal diagonal."""
        plt.figure(figsize=(14, 10))
        max_val = max(max(self.truths), max(self.guesses))

        plt.plot([0, max_val], [0, max_val], color='deepskyblue', lw=3,
                 alpha=0.7, label='Perfect prediction')
        plt.scatter(self.truths, self.guesses, s=20, c=self.colors, alpha=0.6)

        plt.xlabel('Actual Price ($)', fontsize=12)
        plt.ylabel('Predicted Price ($)', fontsize=12)
        plt.xlim(0, max_val)
        plt.ylim(0, max_val)
        plt.title(title, fontsize=14, fontweight='bold')
        plt.grid(alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.show()

    def report(self):
        """Print aggregate metrics and render the evaluation chart."""
        mae      = sum(self.errors) / self.size
        rmsle    = math.sqrt(sum(self.sles) / self.size)
        hits     = sum(1 for c in self.colors if c == "green")
        hit_rate = hits / self.size * 100

        separator = "=" * 60
        print(f"\n{separator}")
        print("EVALUATION SUMMARY")
        print(separator)
        print(f"Model              : {self.title}")
        print(f"Test size          : {self.size}")
        print(f"Mean absolute error: ${mae:,.2f}")
        print(f"RMSLE              : {rmsle:.4f}")
        print(f"Hit rate (green)   : {hit_rate:.1f}%  ({hits}/{self.size})")
        print(f"{separator}\n")

        chart_title = f"{self.title}  |  MAE ${mae:,.2f}  |  RMSLE {rmsle:.3f}  |  Hits {hit_rate:.1f}%"
        self.chart(chart_title)

    def run(self):
        """Iterate over test items, run predictions, and produce a report."""
        print(f"Evaluating {self.size} examples with '{self.title}'...\n")
        for i in tqdm(range(self.size), desc="Running inference"):
            self.run_datapoint(i)
        self.report()

    @classmethod
    def test(cls, function, data, **kwargs):
        """Convenience method: create a Tester and run it in one call."""
        cls(function, data, **kwargs).run()

## Run Evaluation

In [ ]:
# Evaluate the QLoRA fine-tuned model on the test set
Tester.test(predict_price, test, title="LLaMA 3.1 8B + QLoRA Adapters (400K)")